# 03 - Analysis
Exploratory analysis on top of the cleaned analytical tables: core KPIs, MSP performance, weather correlation, forecasting, anomaly detection and clustering.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import sqlite3, pandas as pd
from src import config, analytics
from dashboard import kpis

conn = sqlite3.connect(config.SQLITE_DB_PATH)
arrivals = pd.read_sql_query('SELECT * FROM arrivals_features', conn)
prices = pd.read_sql_query('SELECT * FROM price_features', conn)
weather_daily = pd.read_sql_query('SELECT * FROM weather_daily', conn)
market_value = pd.read_sql_query('SELECT * FROM market_value', conn)
arrivals['arrival_date'] = pd.to_datetime(arrivals['arrival_date'])
prices['price_date'] = pd.to_datetime(prices['price_date'])
weather_daily['date'] = pd.to_datetime(weather_daily['date'])
market_value['arrival_date'] = pd.to_datetime(market_value['arrival_date'])

## Core KPIs

In [2]:
print('Total arrivals (Quintal):', round(kpis.total_arrivals_quintal(arrivals), 2))
print('Total estimated market value:', round(kpis.total_market_value(market_value), 2))
print('Avg modal price:', round(kpis.avg_modal_price(prices), 2))
print('Avg MSP:', round(kpis.avg_msp(prices), 2))
print('Avg MSP gap %:', round(kpis.avg_msp_gap_pct(prices), 2))
print('% observations below MSP:', round(kpis.pct_below_msp(prices), 2))
print('Active Mandis:', kpis.active_mandi_count(arrivals))
print('Active crops:', kpis.active_crop_count(arrivals))
print('Arrival growth rate %:', round(kpis.arrival_growth_rate(arrivals), 2))

Total arrivals (Quintal): 102736327.68
Total estimated market value: 251147690781.64
Avg modal price: 3770.74
Avg MSP: 2378.78
Avg MSP gap %: 60.62
% observations below MSP: 19.14
Active Mandis: 57
Active crops: 7
Arrival growth rate %: -43.47


## Mandi Risk Score

In [3]:
risk_df = kpis.compute_mandi_risk_scores(prices, arrivals, weather_daily)
risk_df.sort_values('risk_score', ascending=False).head(10)

,mandi_id,msp_pressure_raw,price_volatility_raw,arrival_anomaly_raw,weather_anomaly_raw,msp_pressure_score,price_volatility_score,arrival_anomaly_score,weather_anomaly_score,risk_score,risk_category
22,MANDI023,5.890420,1454.514622,5.637320,149.536686,100.000000,100.000000,31.084689,0.0,66.22,High
6,MANDI007,4.896232,1353.887247,13.597449,149.536686,74.934754,60.290038,99.696311,0.0,61.24,High
25,MANDI026,4.641627,1419.881460,10.120648,149.536686,68.515714,86.332928,69.728333,0.0,59.51,Medium
28,MANDI029,4.556825,1364.163180,10.353641,149.536686,66.377692,64.345166,71.736594,0.0,53.67,Medium
47,MANDI048,4.376015,1374.645697,10.269055,149.536686,61.819161,68.481817,71.007517,0.0,52.96,Medium
5,MANDI006,4.476887,1307.592503,12.987583,149.536686,64.362312,42.021027,94.439625,0.0,51.92,Medium
11,MANDI012,4.311372,1348.127820,11.313487,149.536686,60.189395,58.017230,80.009908,0.0,51.57,Medium
18,MANDI019,4.613550,1366.674626,8.411208,149.536686,67.807842,65.336242,54.993969,0.0,51.07,Medium
56,MANDI057,4.018086,1390.126526,9.899266,149.536686,52.795121,74.590922,67.820153,0.0,50.69,Medium
8,MANDI009,4.355251,1276.254624,13.632682,149.536686,61.295655,29.654353,100.000000,0.0,48.87,Medium


## Weather vs Arrivals correlation

In [4]:
daily_arr = arrivals.groupby('arrival_date')['quantity_quintal'].sum().reset_index()
daily_arr = daily_arr.rename(columns={'quantity_quintal':'total_quantity_quintal','arrival_date':'date'})
merged = daily_arr.merge(weather_daily, on='date', how='inner')
merged[['total_rainfall_mm','avg_temperature_c','avg_humidity_pct','total_quantity_quintal']].corr()

,total_rainfall_mm,avg_temperature_c,avg_humidity_pct,total_quantity_quintal
total_rainfall_mm,1.000000,-0.171000,0.103851,0.066139
avg_temperature_c,-0.171000,1.000000,-0.174141,-0.004563
avg_humidity_pct,0.103851,-0.174141,1.000000,0.012634
total_quantity_quintal,0.066139,-0.004563,0.012634,1.000000


## Forecasting (Exponential Smoothing)

In [5]:
crop_example = arrivals['crop'].value_counts().idxmax()
forecast_df = analytics.forecast_arrivals(arrivals, crop=crop_example, periods=14)
forecast_df.tail(14)

,date,forecast,is_forecast
343,2026-12-10,18827.927521,True
344,2026-12-11,18894.057508,True
345,2026-12-12,18960.187495,True
346,2026-12-13,19026.317482,True
347,2026-12-14,19092.447468,True
348,2026-12-15,19158.577455,True
349,2026-12-16,19224.707442,True
350,2026-12-17,19290.837429,True
351,2026-12-18,19356.967416,True
352,2026-12-19,19423.097403,True


## Anomaly Detection (IQR + rolling z-score)

In [6]:
arrival_anomalies = analytics.detect_arrival_anomalies(arrivals)
arrival_anomalies[arrival_anomalies['iqr_anomaly']].head(10)

,arrival_date,crop,quantity_quintal,iqr_anomaly,zscore_anomaly
3,2026-01-04,Basmati,1.514873e+04,True,False
11,2026-01-12,Basmati,1.216758e+04,True,False
26,2026-01-27,Basmati,5.171417e+04,True,False
37,2026-02-07,Basmati,1.141925e+06,True,False
38,2026-02-08,Basmati,5.199609e+05,True,False
44,2026-02-14,Basmati,1.620633e+06,True,False
51,2026-02-21,Basmati,2.555838e+05,True,False
53,2026-02-23,Basmati,1.254125e+06,True,False
68,2026-03-10,Basmati,1.264651e+05,True,False
76,2026-03-18,Basmati,6.198371e+05,True,False


## Mandi Clustering (K-Means)

In [7]:
cluster_df = analytics.cluster_mandis(arrivals, prices, n_clusters=4)
cluster_df.groupby('cluster').agg(mandis=('mandi_id','count'), avg_arrivals=('total_arrivals','mean'), avg_price=('avg_modal_price','mean'))

,mandis,avg_arrivals,avg_price
cluster,,,
0,17,9.168667e+05,3886.976113
1,27,9.415740e+05,3707.427557
2,12,3.882790e+06,3764.284572
3,1,1.513361e+07,3867.089452


These same functions power the interactive Streamlit dashboard (`dashboard/app.py`), so every number a Board member sees on screen can be reproduced here.